# 5.9 — Hidden Markov Models

A hidden state evolves through time, and each observation gives noisy evidence about where it is.

HMMs are dynamic Bayesian networks with one hidden chain and emissions. They use Bayesian filtering, message passing, and lead to Kalman filters and CRFs.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(59)


def normalize(values):
    values = np.asarray(values, dtype=float)
    total = values.sum()
    if total <= 0:
        return np.ones_like(values) / values.size
    return values / total


def row_normalize(values):
    values = np.asarray(values, dtype=float)
    totals = values.sum(axis=1, keepdims=True)
    return values / totals


def forward_backward_viterbi(pi, T, E, obs):
    pi = np.asarray(pi, dtype=float)
    T = np.asarray(T, dtype=float)
    E = np.asarray(E, dtype=float)
    obs = list(obs)
    n_states = len(pi)
    n_time = len(obs)
    alpha = np.zeros((n_time, n_states))
    scales = np.zeros(n_time)
    raw0 = pi * E[:, obs[0]]
    scales[0] = raw0.sum()
    alpha[0] = raw0 / scales[0]
    for t in range(1, n_time):
        raw = E[:, obs[t]] * (alpha[t - 1] @ T)
        scales[t] = raw.sum()
        alpha[t] = raw / scales[t]
    beta = np.ones((n_time, n_states))
    for t in range(n_time - 2, -1, -1):
        raw = T @ (E[:, obs[t + 1]] * beta[t + 1])
        beta[t] = raw / scales[t + 1]
    gamma = alpha * beta
    gamma = gamma / gamma.sum(axis=1, keepdims=True)
    loglik = float(np.log(scales).sum())
    delta = np.zeros((n_time, n_states))
    back = np.zeros((n_time, n_states), dtype=int)
    delta[0] = np.log(pi) + np.log(E[:, obs[0]])
    for t in range(1, n_time):
        scores = delta[t - 1][:, None] + np.log(T)
        back[t] = np.argmax(scores, axis=0)
        delta[t] = np.max(scores, axis=0) + np.log(E[:, obs[t]])
    path = [int(np.argmax(delta[-1]))]
    for t in range(n_time - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    path = list(reversed(path))
    return {"alpha": alpha, "gamma": gamma, "loglik": loglik, "likelihood": float(np.exp(loglik)), "path": path, "scales": scales}


def exact_hmm_marginals(pi, T, E, obs):
    result = forward_backward_viterbi(pi, T, E, obs)
    return result["gamma"]


def make_hmm_ladder():
    d1 = {"name": "D1 2-state HMM", "pi": [0.6, 0.4], "T": [[0.7, 0.3], [0.3, 0.7]], "E": [[0.9, 0.1], [0.2, 0.8]], "obs": [0, 1, 1]}
    d2 = {"name": "D2 4-state HMM", "pi": normalize([3, 2, 1, 1]), "T": row_normalize(rng.uniform(0.2, 1.0, (4, 4))), "E": row_normalize(rng.uniform(0.2, 1.0, (4, 3))), "obs": [0, 1, 2, 1, 0]}
    d3 = {"name": "D3 ambiguous emissions", "pi": normalize([1, 1]), "T": [[0.92, 0.08], [0.08, 0.92]], "E": [[0.55, 0.45], [0.45, 0.55]], "obs": [0, 1, 0, 1, 1, 0]}
    d4 = {"name": "D4 sequence contingency trace", "pi": normalize([30, 20, 10]), "T": normalize(np.array([[18, 5, 2], [4, 16, 6], [3, 5, 14]], dtype=float)), "E": normalize(np.array([[20, 3, 2], [6, 15, 5], [2, 7, 18]], dtype=float)), "obs": [0, 0, 1, 2, 2, 1, 0]}
    d5 = {"name": "D5 ill-conditioned long HMM", "pi": normalize([1, 1, 1]), "T": [[0.985, 0.010, 0.005], [0.010, 0.980, 0.010], [0.005, 0.010, 0.985]], "E": [[0.36, 0.34, 0.30], [0.34, 0.36, 0.30], [0.33, 0.33, 0.34]], "obs": [0, 1, 0, 1, 2, 2, 1, 0, 0, 1, 2, 1, 0, 2, 2]}
    return [d1, d2, d3, d4, d5]


## The concept, built once (D1)

The forward message is

$$\alpha_t(s)=p(y_{1:t},z_t=s)=p(y_t\mid s)\sum_r\alpha_{t-1}(r)p(s\mid r).$$

The lesson's first filter normalizes $[0.6\cdot0.9,0.4\cdot0.2]=[0.54,0.08]$ to $[0.871,0.129]$.

In [ ]:

pi = np.array([0.6, 0.4])
T = np.array([[0.7, 0.3], [0.3, 0.7]])
E = np.array([[0.9, 0.1], [0.2, 0.8]])
obs = [0, 1, 1]
first_raw = pi * E[:, obs[0]]
first_filter = normalize(first_raw)
print("first raw", first_raw)
print("first filter", first_filter)
assert np.allclose(first_raw, [0.54, 0.08])
assert np.allclose(np.round(first_filter, 3), [0.871, 0.129])


Forward-backward keeps scale factors for likelihood, while Viterbi uses max-product backpointers for a single best path rather than marginal states.

In [ ]:

result = forward_backward_viterbi(pi, T, E, obs)
print("likelihood", result["likelihood"])
print("path", result["path"])
print("smoothed initial", result["gamma"][0])
assert np.isclose(result["likelihood"], 0.131542)
assert result["path"] == [0, 1, 1]
assert result["gamma"][0, 1] > result["alpha"][0, 1]


## The dataset ladder

D1-D5 increase state count, ambiguity, sequence length, and numerical conditioning.

In [ ]:

ladder = make_hmm_ladder()
for rung in ladder:
    print(rung["name"], "states", len(rung["pi"]), "length", len(rung["obs"]), "obs", rung["obs"][:8])


In [ ]:

rows = []
for rung in ladder:
    result = forward_backward_viterbi(rung["pi"], rung["T"], rung["E"], rung["obs"])
    exact = exact_hmm_marginals(rung["pi"], rung["T"], rung["E"], rung["obs"])
    error = float(np.max(np.abs(result["gamma"] - exact)))
    rows.append({"name": rung["name"], "gamma": result["gamma"], "exact": exact, "error": error, "path": result["path"], "scales": result["scales"]})
for row in rows:
    print(row["name"], "marginal error", f"{row['error']:.6f}", "path prefix", row["path"][:6])


In [ ]:

fig, axes = plt.subplots(2, len(rows), figsize=(16, 6))
for j, row in enumerate(rows):
    axes[0, j].imshow(row["gamma"].T, aspect="auto", vmin=0, vmax=1)
    axes[0, j].set_title(row["name"].split()[0])
    axes[0, j].set_ylabel("state")
    time_error = np.max(np.abs(row["gamma"] - row["exact"]), axis=1)
    axes[1, j].plot(time_error, marker="o")
    axes[1, j].set_title("marginal error vs time")
    axes[1, j].set_xlabel("time")
fig.suptitle("Estimated vs exact state marginals and error curve")
plt.tight_layout()


## Pitfall on the hardest rung

If you normalize forward messages but throw away the scale factors, you lose the sequence likelihood. Also, Viterbi states are not marginal argmax states.

In [ ]:

d5 = ladder[-1]
result = forward_backward_viterbi(d5["pi"], d5["T"], d5["E"], d5["obs"])
wrong_likelihood = 1.0
fixed_likelihood = float(np.prod(result["scales"]))
marginal_states = np.argmax(result["gamma"], axis=1).tolist()
print("wrong likelihood after forgetting scales", wrong_likelihood)
print("fixed likelihood from scales", fixed_likelihood)
print("Viterbi prefix", result["path"][:8])
print("marginal argmax prefix", marginal_states[:8])
assert fixed_likelihood < 1.0
assert fixed_likelihood > 0.0


## Evaluate it + Practice

- Metric: maximum marginal error versus exact forward-backward
- No-skill baseline: use the initial distribution at every time
- Cheap sanity check: first filter equals [0.871, 0.129]
- Ablation: drop scale factors and likelihood becomes unusable
- Failure signals: underflow, Viterbi/marginal disagreement, or smoothing needed but filtering used


Practice: Change D3 emissions to be identical and inspect uncertainty.

Practice: Compare Viterbi states to marginal argmax states on D5.

Practice: Compute log-likelihood from scale factors by hand for D1.